# NanoChat — Full Training Pipeline

Covers the full pipeline: tokenizer → pretraining → SFT → eval → chat.

Tuned for a single T4/A100 GPU (16–40GB VRAM). Adjust `DEPTH` and `BATCH_SIZE` below if you're on something smaller or larger.

> **Before running**: make sure the GPU accelerator is enabled (Runtime → Change runtime type → GPU).

## 0. Environment check

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU found. Enable GPU acceleration before continuing.")

print("GPU:", result.stdout.strip())

## 1. Setup

In [ ]:
# Clone the repo (skip if already present)
import os

REPO_URL = "https://github.com/bouchnam/nanochat-safety.git"
REPO_DIR = "/kaggle/working/nanochat-safety"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Install dependencies
# torch is already available on Kaggle/Colab, so we skip it and install the rest
!pip install -q \
    datasets>=4.0.0 \
    transformers>=4.57.3 \
    tiktoken>=0.11.0 \
    tokenizers>=0.22.0 \
    rustbpe>=0.1.0 \
    regex>=2025.9.1 \
    wandb>=0.21.3 \
    zstandard>=0.25.0 \
    tabulate>=0.9.0 \
    scipy>=1.15.3 \
    psutil>=7.1.0

print("Dependencies installed.")

In [ ]:
import os
import torch

# Paths
BASE_DIR = "/kaggle/working/nanochat-cache"
os.makedirs(BASE_DIR, exist_ok=True)
os.environ["NANOCHAT_BASE_DIR"] = BASE_DIR
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["WANDB_RUN"] = "dummy"  # set to your wandb run name if you want logging

# Force float16 on T4 (Turing, no native bf16). Use bfloat16 on A100/H100.
capability = torch.cuda.get_device_capability()
if capability[0] >= 8:
    os.environ["NANOCHAT_DTYPE"] = "bfloat16"
else:
    os.environ["NANOCHAT_DTYPE"] = "float16"

print(f"CUDA {capability[0]}.{capability[1]} → dtype: {os.environ['NANOCHAT_DTYPE']}")

# ── Model / training config ────────────────────────────────────────────────────
# depth=12 ~ 73M params, fits comfortably on T4 (16GB) with batch_size=16
# depth=20 ~ 300M params, better quality, needs A100
DEPTH       = 12
BATCH_SIZE  = 16   # per-GPU
SEQ_LEN     = 1024
N_SHARDS    = 32   # number of data shards to download (~3.2GB, ~800M tokens)
# ──────────────────────────────────────────────────────────────────────────────

## 2. Data

In [ ]:
# Download data shards. Each shard is ~100MB compressed (~250M chars of text).
# 32 shards ≈ 3.2GB on disk, ~800M tokens — enough for a decent small model run.
!python -m nanochat.dataset -n {N_SHARDS}

## 3. Tokenizer

In [ ]:
# Train a BPE tokenizer on the first ~2B characters of data.
# Vocabulary size: 32,768 (2^15). GPT-4 style split pattern.
# Takes ~1 min on CPU.
!python -m scripts.tok_train --max-chars=2_000_000_000

In [ ]:
# Sanity check: compare compression ratio against GPT-2 and GPT-4 tokenizers.
!python -m scripts.tok_eval

## 4. Pretraining

In [ ]:
# Single-GPU training. torchrun with nproc_per_node=1 is intentional —
# it keeps the DDP code path active for consistency with multi-GPU runs.
#
# Notable flags:
#   --depth               controls model size; everything else scales from it
#   --target-param-data-ratio  Chinchilla ratio (default 10.5); lower = faster but less quality
#   --core-metric-every=-1     skip CORE eval during training (expensive, run it separately)

!torchrun --standalone --nproc_per_node=1 -m scripts.base_train -- \
    --depth={DEPTH} \
    --device-batch-size={BATCH_SIZE} \
    --max-seq-len={SEQ_LEN} \
    --core-metric-every=-1 \
    --sample-every=500 \
    --run=$WANDB_RUN

## 5. Base model eval

In [ ]:
# Evaluate bits-per-byte on train/val splits and draw some samples.
# Skipping CORE metric here (it takes ~20 min); uncomment if you want it.
!torchrun --standalone --nproc_per_node=1 -m scripts.base_eval -- \
    --device-batch-size={BATCH_SIZE} \
    --split-tokens=524288 \
    --max-per-task=32
    # --core-metric   # uncomment to run CORE

## 6. SFT (Supervised Fine-Tuning)

Teaches the model conversation format, tool use, and multiple choice reasoning.
Training data mixture: MMLU + GSM8K + SmolTalk + a small identity dataset.

In [ ]:
import os

# Identity conversations: gives the model a name and basic self-knowledge.
# Swap this out (or extend it) with your own JSONL to customise the personality.
identity_path = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "identity_conversations.jsonl")
if not os.path.exists(identity_path):
    !curl -sL -o {identity_path} \
        https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl
    print("Downloaded identity conversations.")
else:
    print("Already present, skipping download.")

In [ ]:
!torchrun --standalone --nproc_per_node=1 -m scripts.chat_sft -- \
    --device-batch-size={BATCH_SIZE} \
    --max-seq-len={SEQ_LEN} \
    --run=$WANDB_RUN

## 7. SFT eval

In [ ]:
!torchrun --standalone --nproc_per_node=1 -m scripts.chat_eval -- -i sft

## 8. Checkpoint

Download the trained model checkpoint before the session ends.

In [ ]:
import shutil, os

base_dir = os.environ["NANOCHAT_BASE_DIR"]
ckpt_src = os.path.join(base_dir, "checkpoints")

if os.path.exists(ckpt_src):
    archive = shutil.make_archive("/kaggle/working/nanochat_checkpoint", "zip", ckpt_src)
    print(f"Checkpoint saved to: {archive}")
    print(f"Size: {os.path.getsize(archive) / 1e6:.1f} MB")
else:
    print("No checkpoint directory found at", ckpt_src)

## 9. Quick chat test

In [ ]:
prompts = [
    "What is the capital of France?",
    "Explain what a transformer is in one paragraph.",
    "Write a Python function that checks if a number is prime.",
]

for p in prompts:
    print(f">>> {p}")
    !python -m scripts.chat_cli -p "{p}"
    print()